# GeoTessera: Workshop Demo

This notebook is a tour of [`geotessera`](https://github.com/ucam-eo/geotessera), the Python package for getting hold of Tessera embeddings.

We will:

1. Install `geotessera` into a conda environment.
2. Look at how the embeddings are stored, and read them **without** `geotessera`, using nothing but `xarray` + `zarr`.
3. Tour the key functions of the **streaming** (Zarr) interface, `GeoTesseraZarr`.
4. Tour the **non-streaming** (NumPy tile / GeoTIFF) interface, `GeoTessera`, and compare the two.
5. Run a simple k-means clustering over a bit of Cambridge.

Most cells read data over the network from a public bucket. Everything is kept to small areas, so the whole notebook should run in a few minutes and download a few hundred MB at most (most of that is one NumPy tile in Part 4).

## 0. Setup

Imports

In [ ]:
import time
from pathlib import Path

import numpy as np
import xarray as xr
import zarr
import matplotlib.pyplot as plt
import geotessera

print("geotessera", geotessera.__version__)
print("xarray    ", xr.__version__)
print("zarr      ", zarr.__version__)


Areas of interest

In [ ]:
YEAR = 2024

# Central Cambridge: city centre, the Backs, Grantchester Meadows, Coldham's Common...
CAMBRIDGE = (0.09, 52.185, 0.15, 52.225)
# A small box around the city centre, for quick reads
SMALL = (0.11, 52.200, 0.13, 52.210)
# Straddles the Greenwich meridian, which is the boundary between UTM zones 30 and 31
SEAM = (-0.05, 52.18, 0.05, 52.22)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

Various helper functions

In [ ]:
from sklearn.decomposition import PCA


def extent(transform, shape):
    # matplotlib imshow extent (left, right, bottom, top) for a north-up raster.
    h, w = shape[:2]
    return (transform.c, transform.c + w * transform.a, transform.f + h * transform.e, transform.f)


def rgb_limits(arr3, pct=(2, 98)):
    # Per-channel percentile limits, shape (2, 3), over the valid pixels.
    valid = np.isfinite(arr3).all(axis=-1)
    return np.percentile(arr3[valid], pct, axis=0)


def to_rgb(arr3, limits=None):
    # (H, W, 3) floats -> (H, W, 4) RGBA in [0, 1]; pixels with no data become transparent.
    valid = np.isfinite(arr3).all(axis=-1)
    lo, hi = rgb_limits(arr3) if limits is None else limits
    rgb = np.clip((arr3 - lo) / (hi - lo), 0, 1)
    return np.dstack([np.nan_to_num(rgb), valid]).astype(np.float32)


def fit_pca(emb, n=3, max_samples=100_000, seed=0):
    # Fit a PCA on (a subsample of) the valid pixels of an (H, W, 128) embedding.
    X = emb[np.isfinite(emb).all(axis=-1)]
    rng = np.random.default_rng(seed)
    if len(X) > max_samples:
        X = X[rng.choice(len(X), max_samples, replace=False)]
    return PCA(n_components=n, random_state=seed).fit(X)


def pca_project(emb, pca):
    # Project every valid pixel onto the PCA components; NaN elsewhere.
    valid = np.isfinite(emb).all(axis=-1)
    out = np.full(emb.shape[:2] + (pca.n_components_,), np.nan, np.float32)
    out[valid] = pca.transform(emb[valid])
    return out


def pca_rgb(emb):
    return to_rgb(pca_project(emb, fit_pca(emb)))


def show(img, transform, ax=None, title=None, **kw):
    ax = ax or plt.gca()
    ax.imshow(img, extent=extent(transform, img.shape), interpolation="nearest", **kw)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
    return ax

## 1. Installation (conda)

`geotessera` installs from PyPI with pip,

Ideally do this *inside* a conda environment. For example:

```bash
conda create --name my-tessera-env -c conda-forge python=3.14 numpy xarray rasterio geopandas matplotlib scipy ipykernel jupyter pip
conda activate my-tessera-env
pip install geotessera
```

## 2. Reading the embeddings *without* geotessera

It's worth seeing that there's no magic. The embeddings are contained in a "normal" Zarr store: a public, self-describing dataset that any Zarr-aware tool can read. Doing it by hand once makes it much clearer what `geotessera` does for you.

There is one group per UTM zone, `utm01` ... `utm60`. Cambridge is in zone 31.

In [ ]:
zone = 31

STORE_URL = (
    "https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop"
    "/tessera/tessera/zarr/v1.1-dclimate"
)

ds = xr.open_zarr(STORE_URL, group="utm31", consolidated=True, chunks=None)

In [ ]:
ds

Things to notice:
* There are arrays for `embeddings` and `scales`, the "actual" embeddings are given by the product of these two.
* The data have been "lazily" loaded into `dask` arrays. Nothing has been downloaded yet, the data can be streamed when we need it.
* `time` is the year, `x` / `y` are **pixel-centre** eastings / northings in metres, in the zone's CRS (`proj:code` attribute).
* There are also quality-assurance variables: how many Sentinel-1/2 observations went into each pixel (`s2_obs_count`, ...), and which months had coverage.

To read a region we project the lon/lat box into the zone's CRS. We use `transform_bounds` with densified edges, not just the two corners, because a lon/lat rectangle is a slightly rotated, curved quadrilateral on the UTM grid (more on this in Part 7).

In [ ]:
from pyproj import Transformer

# transform the "SMALL" bounding box into the UTM zone 31 CRS
crs = 'EPSG:32631'
to_utm = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
west, south, east, north = to_utm.transform_bounds(*SMALL, densify_pts=21)

# slice the array to get the embeddings and scales for the given bounding box and year
window = ds[["embeddings", "scales"]].sel(time=YEAR, x=slice(west, east), y=slice(north, south))

# print the sizes of the arrays in the window
print(dict(window.sizes))

# download the actual embeddings and scales
emb_int8 = window["embeddings"].values
scales = window["scales"].values

print("embeddings", emb_int8.shape, emb_int8.dtype)
print("scales", scales.shape, scales.dtype)

Now dequantise by hand: multiply each pixel's 128 `int8` values by its scale, mark non-finite scales as no-data, and move the band axis last to get the conventional `(H, W, 128)` layout.

In [ ]:
emb = emb_int8 * scales
emb = emb.transpose(1, 2, 0)

print("shape", emb.shape, emb.dtype)
print(f"value range: {np.min(emb):.2f} .. {np.max(emb):.2f}")

Make RGB plots

In [ ]:
# get the "transform" of the window, from the pixel-centre coordinates
from rasterio.transform import Affine
px = float(ds.attrs["spatial:transform"][0])
transform = Affine(px, 0, window.x.values[0] - px / 2, 0, -px, window.y.values[0] + px / 2)

# make RGB plots
fig, axs = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, bands in zip(axs, [(0, 1, 2), (10, 50, 100), (3, 64, 127)]):
    show(to_rgb(emb[..., list(bands)]), raw_transform, ax=ax, title=f"bands {bands}")
plt.tight_layout()

First fit a (3-component) PCA to the embeddings, then make an RGB plot from the 3 principal components

In [ ]:
pca = fit_pca(emb)
print("variance explained by 3 PCs:", pca.explained_variance_ratio_.round(3), "total", pca.explained_variance_ratio_.sum().round(3))

pcs_2024 = pca_project(emb, pca)
limits = rgb_limits(pcs_2024)

fig, ax = plt.subplots(figsize=(9, 9))
show(to_rgb(pcs_2024, limits), transform, ax=ax, title=f"Cambridge {YEAR}: first 3 principal components as RGB")
plt.show()

## 3. Streaming with `GeoTesseraZarr`

`GeoTesseraZarr` reads from the Zarr store. Like we saw above, nothing is downloaded up front, and each call fetches only the chunks it needs.

In [ ]:
from geotessera import GeoTesseraZarr

gz = GeoTesseraZarr()          # default: the v1.1 'dclimate' store


In [ ]:
gz

### 3.1 `read_region`: a lon/lat box → `(H, W, 128)` array

It returns `(mosaic, transform, crs)`. The mosaic is on the native grid of the zone holding the bbox centre. Note: the dequantisation (int8 * scales) is already done under the hood.

In [ ]:
small, small_transform, small_crs = gz.read_region(SMALL, year=YEAR)
print(small.shape, small.dtype, small_crs)


Now the main Cambridge area, which we reuse for the clustering section below. 4 km × 4.5 km is ~180,000 pixels, about 90 MB as float32.

In [ ]:
emb, transform, crs = gz.read_region(CAMBRIDGE, year=YEAR)
print(emb.shape, crs)

### 3.2 `sample_points`: embeddings at a list of `(lon, lat)` points

This is a vectorised read, one bulk request per zone. It is ideal for building training sets from labelled points. A few Cambridge landmarks:

In [ ]:
places = {
    "King's College Chapel": (0.1164, 52.2042),
    "Market Square":         (0.1185, 52.2056),
    "Parker's Piece":        (0.1250, 52.2017),
    "Jesus Green":           (0.1215, 52.2105),
    "Midsummer Common":      (0.1290, 52.2100),
    "Grantchester Meadows":  (0.1050, 52.1900),
    "Farmland near Coton":   (0.0700, 52.2080),
    "River Cam (Backs)":     (0.1148, 52.2036),
}
X = gz.sample_points(list(places.values()), year=YEAR)
print(X.shape)


### 3.3 `probe`: *why* is there no value?

`sample_points` gives a NaN row wherever there's no embedding. `probe` tells you why: `'valid'`, `'water'`, `'nodata'` (inside the grid, but never written) or `'outside'` (beyond the zone's grid).

In [ ]:
for name, (lon, lat) in {"King's College": (0.1164, 52.2042), "mid-Atlantic": (-30.0, 40.0)}.items():
    vec, status = gz.probe(lon, lat, year=YEAR)
    print(f"{name:15s} -> {status}")

### 3.4 `iter_region`: stream a region strip by strip

For regions too big for memory, `iter_region` yields `(block, transform, crs)` strips from top to bottom, and downloads the next strip while you work on the current one. Here we compute the mean embedding of the Cambridge box without ever holding all of it at once.

In [ ]:
total, count = np.zeros(gz.n_bands), 0
for block, block_transform, _ in gz.iter_region(CAMBRIDGE, year=YEAR, strip_rows=64):


    print(f"strip {block.shape} starting at N={block_transform.f:,.0f}")

    # do some calculations for the block, e.g. take the mean of the block
    m = block.mean()
    print(f"strip mean: {m}")


### 3.5 `read_patch`: a fixed-size patch centred on a point

This method gets a square patch, e.g. 256 × 256 pixels, around a specified point. The specified point is always at pixel `[size_px // 2, size_px // 2]`, and 



In [ ]:
patch, patch_transform, patch_crs = gz.read_patch(0.1164, 52.2042, year=YEAR, size_px=256)
print(patch.shape, patch_crs)

A key difference between `read_region` and `read_patch` is that the `read_patch` gives you something **merged across UTM zones** if it straddles a boundary. `read_region` doesn't merge zones for you!

We can see the difference between `read_region` and `read_patch` by looking at their outputs across a UTM seam:

In [ ]:
# setting up a separate GeoTesseraZarr instance for the v1.1 "cambridge" variant, which has even sharper boundary issues
gz_cam = GeoTesseraZarr(store_url='https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/tessera/tessera/zarr/v1.1')

# loading embeddings via read_region
seam_region, seam_t, seam_crs = gz_cam.read_region(SEAM, year=YEAR)
print("read_region:", seam_region.shape, seam_crs, f"| {np.isnan(seam_region).all(-1).mean():.0%} of pixels are NaN")

# loading embeddings via read_patch
seam_patch, patch_t, patch_crs_wkt = gz_cam.read_patch(0.0, 52.2, year=YEAR, size_px=600)
print("read_patch: ", seam_patch.shape, f"| {np.isnan(seam_patch).all(-1).mean():.0%} NaN | CRS:", patch_crs_wkt[:60], "...")

# plot both
seam_pca = fit_pca(seam_patch)
seam_limits = rgb_limits(pca_project(seam_patch, seam_pca))
fig, axs = plt.subplots(1, 2, figsize=(15, 6.5))
show(to_rgb(pca_project(seam_region, seam_pca), seam_limits), seam_t, ax=axs[0], title=f"read_region({SEAM}) in {seam_crs}")
show(to_rgb(pca_project(seam_patch, seam_pca), seam_limits), patch_t, ax=axs[1], title="read_patch(0.0, 52.2, size_px=600), merged")
x_m, y_m = Transformer.from_crs("EPSG:4326", seam_crs, always_xy=True).transform([0, 0], [SEAM[1], SEAM[3]])
axs[0].plot(x_m, y_m, "r--", label="0° meridian")
axs[0].legend()
axs[1].axvline(500_000, color="r", ls="--", label="0° meridian")
axs[1].legend()
for ax in axs:
    ax.set_facecolor("0.85")
plt.tight_layout()

### 3.6 `export_geotiffs`: write GeoTIFFs for QGIS/GDAL

This writes one float32 GeoTIFF **per UTM zone** the bbox touches, named `tessera_YEAR_utmNN.tif`, on the native grid, with metadata tags.

`dry_run=True` estimates sizes without downloading anything.

In [ ]:
print(gz.export_geotiffs(CAMBRIDGE, YEAR, DATA_DIR / "exports", dry_run=True))


Let's actually do the real export now, but just write the first 3 bands.

In [ ]:
tifs = gz.export_geotiffs(SMALL, YEAR, DATA_DIR / "exports", bands=[0, 1, 2])
print(tifs)


## 4. Non-streaming: `GeoTessera` and NumPy tiles

The original access route is to **download whole 0.1° × 0.1° tiles** as `.npy` files to a local directory and work from those. Each tile is named by its centre, which always ends in `.x5` (e.g. `grid_0.15_52.25` covers 0.10–0.20°E, 52.20–52.30°N), and consists of:

* `grid_<lon>_<lat>.npy`: `(H, W, 128)` `int8` (~100 MB per tile per year at this latitude);
* `grid_<lon>_<lat>_scales.npy`: `(H, W)` `float32`;
* a small *landmask* GeoTIFF for the tile, which supplies its CRS and geotransform.

NumPy tiles are only published for some datasets. For `v1.1` that's the `cambridge` variant, so we ask for it explicitly. Leaving it out still works, but logs a warning that it doesn't match the default `dclimate` streaming store.

In [ ]:
from geotessera import GeoTessera, dequantize_embedding

gt = GeoTessera(dataset_version="v1.1", dataset_variant="cambridge", embeddings_dir=DATA_DIR / "tiles")

Our Cambridge box touches 4 tiles, about 400 MB of downloads for ~25 MB of embeddings! To keep things light we fetch just one tile (the one with the city centre north of 52.20°N).

`fetch_embedding` downloads the tile into `embeddings_dir` (if not already there), dequantises it, and returns it with its CRS and transform.

In [ ]:
tile, tile_crs, tile_transform = gt.fetch_embedding(0.15, 52.25, year=YEAR)
print(tile.shape, tile.dtype, tile_crs)
print(tile_transform)


This has written a bunch of .npy files to disk:

In [ ]:

for f in sorted((DATA_DIR / "tiles").rglob("*.*")):
    print(f"{f.stat().st_size / 1e6:8.1f} MB  {f}")

The files are ordinary NumPy arrays, so you can also read them without `geotessera` at all.

## 5. k-means clustering over Cambridge

Because similar places have similar embeddings, even a simple unsupervised method like **k-means** on the raw 128-d vectors gives a sensible land-cover-like map, with no labels needed.

We fit on a random subsample of pixels (fast) and then assign every pixel to its nearest cluster centre. First, a quick look at how the within-cluster spread falls with *k* (the "elbow" plot):

In [ ]:
from sklearn.cluster import KMeans
from matplotlib.colors import ListedColormap

valid = np.isfinite(emb).all(axis=-1)
X_all = emb.reshape(-1, emb.shape[-1])
rng = np.random.default_rng(0)
X_fit = X_all[rng.choice(len(X_all), 30_000, replace=False)]

K = 4
km = KMeans(n_clusters=K, n_init=5, random_state=0).fit(X_fit)

labels = np.full(valid.shape, -1)
labels[valid] = km.predict(X_all)
label_img = np.ma.masked_less(labels, 0)


cmap = ListedColormap(plt.cm.okabe_ito.colors[:K])

fig, axs = plt.subplots(1, 2, figsize=(17, 8))
show(label_img, transform, ax=axs[0], title=f"k-means, k={K}", cmap=cmap, vmin=-0.5, vmax=K - 0.5)
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(i)) for i in range(K)]
fracs = np.bincount(labels[valid], minlength=K) / valid.sum()
axs[0].legend(handles, [f"cluster {i} ({f:.0%})" for i, f in enumerate(fracs)], loc="lower left", fontsize=8)

# Satellite imagery underneath, to help interpret the clusters (needs internet access)
ax = axs[1]
ax.set_xlim(extent(transform, emb.shape)[:2])
ax.set_ylim(extent(transform, emb.shape)[2:])
try:
    import contextily as ctx
    ctx.add_basemap(ax, crs=crs, source=ctx.providers.Esri.WorldImagery)
    ax.set_title("Esri World Imagery")
except Exception as err:
    show(to_rgb(pcs_2024, limits), transform, ax=ax, title=f"PCA RGB (basemap unavailable: {type(err).__name__})")
ax.ticklabel_format(style="plain", useOffset=False)
plt.tight_layout()